In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [5]:
code = 'BLLG'
market = 'CN'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [6]:
bllg_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
bllg_timeseries_df

取得件数: 1719


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1204937,BLLG,CN,2019-07-05,0.0650,0.0650,0.0650,0.0650,8000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1204938,BLLG,CN,2019-07-08,0.0675,0.0675,0.0675,0.0675,4000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1204939,BLLG,CN,2019-07-09,0.0675,0.0675,0.0675,0.0675,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1204940,BLLG,CN,2019-07-10,0.0675,0.0675,0.0675,0.0675,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1204941,BLLG,CN,2019-07-11,0.0675,0.0675,0.0675,0.0675,0,0.067,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1714,1206651,BLLG,CN,2026-05-05,0.8000,0.8100,0.7400,0.7500,329541,0.746,...,0.797614,0.739186,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1715,1206652,BLLG,CN,2026-05-06,0.8100,0.8200,0.7900,0.7900,264770,0.750,...,0.799257,0.740743,False,NaN,NaN,-0.01518,NaN,NaN,NaN,False
1716,1206653,BLLG,CN,2026-05-07,0.7900,0.8400,0.7900,0.8100,338139,0.762,...,0.798233,0.740967,False,NaN,NaN,NaN,NaN,10.0,NaN,False
1717,1206654,BLLG,CN,2026-05-08,0.8600,0.8800,0.8000,0.8000,428724,0.774,...,0.800047,0.741553,True,0.774,NaN,NaN,NaN,NaN,NaN,False


In [7]:
def stock_prices_and_gold_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_gold: pd.DataFrame | None = None,
        df_silver: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # 金価格を統合
    if df_gold is not None:
        df_gold_tmp = df_gold.copy() if df_gold is not None else pd.DataFrame()
        if "date" not in df_gold_tmp.columns:
            df_gold_tmp = df_gold_tmp.reset_index()
        df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
        df_gold_tmp = df_gold_tmp.set_index("date")
        df_gold_tmp = df_gold_tmp.loc[start:end]

    # 銀価格を統合
    if df_silver is not None:
        df_silver_tmp = df_silver.copy() if df_silver is not None else pd.DataFrame()
        if "date" not in df_silver_tmp.columns:
            df_silver_tmp = df_silver_tmp.reset_index()
        df_silver_tmp["date"] = pd.to_datetime(df_silver_tmp["date"])
        df_silver_tmp = df_silver_tmp.set_index("date")
        df_silver_tmp = df_silver_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_gold is not None:
        df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
        df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)
    if df_silver is not None:
        df["MA5_SILVER"] = df_silver_tmp["ma5"].reindex(df.index)
        df["MA25_SILVER"] = df_silver_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- GOLD（右軸） ---
    if df_gold is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_GOLD"],
                name="GOLD_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_GOLD"],
                name="GOLD_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- SILVER（左軸） ---
    if df_silver is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SILVER"],
                name="SILVER_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SILVER"],
                name="SILVER_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [8]:
name = "Silverco Mining Ltd"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime('%Y-%m-%d')
# グラフ領域の作成
fig = stock_prices_and_gold_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_gold=None,
    df_silver=None
)
fig.show()

取得件数: 547


In [9]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://bluelagoonresources.com/wp-content/uploads/2020/09/Blue-Lagoon-Pellaire-Technical-Report.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/Blue-Lagoon-Pellaire-Technical-Report.pdf.md


'/workspace/data/Blue-Lagoon-Pellaire-Technical-Report.pdf.md'

In [11]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://bluelagoonresources.com/wp-content/uploads/2022/02/ACS_Dome_Mountain_TR_2021_FINAL.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/ACS_Dome_Mountain_TR_2021_FINAL.pdf.md


'/workspace/data/ACS_Dome_Mountain_TR_2021_FINAL.pdf.md'

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path